In [1]:
from pathlib import Path
import os

# ---------- locate project root ----------
def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if (parent / "Main data").exists():
            return parent
    raise RuntimeError("Project root (CODE UPLOAD) not found.")

PROJECT_ROOT = find_project_root()

# ---------- relative base path ----------
base_path = PROJECT_ROOT / "shanghai_data_save"

# 批次类别
batch_types = ['weekday_batches', 'weekend_batches', 'holiday_batches']

# 子文件夹名称（1-5）
subfolders = ['1', '2', '3', '4', '5']

# 存储所有结果a
all_dirs = {}

for batch_type in batch_types:
    for subfolder in subfolders:
        current_path = os.path.join(base_path, batch_type, subfolder)
        if os.path.exists(current_path):
            # 仅保留文件夹（修复海象表达式问题）
            dirs = []
            for name in os.listdir(current_path):
                full_path = os.path.join(current_path, name)
                if os.path.isdir(full_path):
                    dirs.append(full_path)
            all_dirs[f"{batch_type}/{subfolder}"] = dirs
        else:
            all_dirs[f"{batch_type}/{subfolder}"] = []

# 输出结果样例
for key, folders in all_dirs.items():
    print(f"{key} 包含的文件夹有：")
    for folder in folders:
        print(f"  {folder}")


weekday_batches/1 包含的文件夹有：
weekday_batches/2 包含的文件夹有：
weekday_batches/3 包含的文件夹有：
weekday_batches/4 包含的文件夹有：
weekday_batches/5 包含的文件夹有：
weekend_batches/1 包含的文件夹有：
  C:\DCLASS\myJupyter\code upload\shanghai_data_save\weekend_batches\1\global_avg
  C:\DCLASS\myJupyter\code upload\shanghai_data_save\weekend_batches\1\local_avg
weekend_batches/2 包含的文件夹有：
weekend_batches/3 包含的文件夹有：
weekend_batches/4 包含的文件夹有：
weekend_batches/5 包含的文件夹有：
holiday_batches/1 包含的文件夹有：
holiday_batches/2 包含的文件夹有：
holiday_batches/3 包含的文件夹有：
holiday_batches/4 包含的文件夹有：
holiday_batches/5 包含的文件夹有：


In [2]:
# 提取包含 'local_avg' 的文件名
local_avg_files = []

for key, folders in all_dirs.items():
    print(f"{key} 包含的文件夹有：")
    for folder in folders:
        folder_name = os.path.basename(folder)
        print(f"  检查文件夹: {folder_name}")
        if folder_name == 'local_avg':
            local_avg_files.append(folder)
        

# 输出结果
print("\n包含 'local_avg' 的文件：")
for file in local_avg_files:
    print(file)

weekday_batches/1 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg
weekday_batches/2 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg
weekday_batches/3 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg
weekday_batches/4 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg
weekday_batches/5 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg
weekend_batches/1 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg
weekend_batches/2 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg
weekend_batches/3 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg
weekend_batches/4 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg
weekend_batches/5 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg
holiday_batches/1 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg
holiday_batches/2 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg
holiday_batches/3 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg
holiday_batches/4 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg
holiday_batches/5 包含的文件夹有：
  检查文件夹: global_avg
  检查文件夹: local_avg

包含 'local

In [3]:
import os
import re

# 收集所有 local_avg 文件夹下的文件名（不含路径）
all_filenames = []

for folder in local_avg_files:
    if os.path.exists(folder):
        for fname in os.listdir(folder):
            if os.path.isfile(os.path.join(folder, fname)):
                all_filenames.append(fname)

# 提取唯一文件名
unique_filenames = list(set(all_filenames))

# 提取数字部分用于排序
def extract_number(filename):
    match = re.search(r'_(\d+)\.parquet$', filename)
    return int(match.group(1)) if match else float('inf')

# 排序
sorted_filenames = sorted(unique_filenames, key=extract_number)

# # 输出结果
# print("\n所有 local_avg 文件夹下的唯一文件名（按数字排序）：")
# for name in sorted_filenames:
#     print(name)

# 打印总数量
print(f"\n总文件数：{len(sorted_filenames)}")



总文件数：32092


In [ ]:
import os
import pandas as pd
import re

# 已有的 sorted_filenames 和 local_avg_files
# 假设这两个变量已经在环境中准备好

# ---------- relative output path ----------
save_dir = PROJECT_ROOT / "shanghai_data_save" / "local_grid_time-series"
save_dir.mkdir(parents=True, exist_ok=True)

# 合并每个 parquet 文件
for filename in sorted_filenames:
    dataframes = []

    # 在所有 local_avg 文件夹中查找此文件
    for folder in local_avg_files:
        full_path = os.path.join(folder, filename)
        if os.path.exists(full_path):
            try:
                df = pd.read_parquet(full_path)
                dataframes.append(df)
            except Exception as e:
                print(f"⚠️ 读取失败: {full_path}，错误信息: {e}")
    
    # 合并并保存
    if dataframes:
        try:
            combined_df = pd.concat(dataframes, ignore_index=True)
            output_path = os.path.join(save_dir, filename)
            combined_df.to_parquet(output_path, index=False)
            # print(f"✅ 合并保存: {output_path}")
        except Exception as e:
            print(f"❌ 合并或保存失败: {filename}，错误信息: {e}")
    else:
        print(f"⚠️ 未找到任何 '{filename}' 的数据，已跳过。")



In [5]:
import pandas as pd

# ---------- relative file path ----------
file_path = (
    PROJECT_ROOT
    / "shanghai_data_save"
    / "local_grid_time-series"
    / "honeycomb_24102.parquet"
)
# 读取 parquet 文件
try:
    df = pd.read_parquet(file_path)
    print("✅ 读取成功，前几行如下：")
    print(df.head())
    print(f"\n📏 数据行数: {len(df)}")

except Exception as e:
    print(f"❌ 读取失败，错误信息: {e}")


✅ 读取成功，前几行如下：
        batch  local_avg_speed  honeycomb_name
0  batch_1730           19.748           24102
1  batch_1733           22.420           24102
2  batch_1741           19.202           24102
3  batch_1751           19.042           24102
4  batch_1760           20.207           24102

📏 数据行数: 203


In [8]:
import os
import pandas as pd
import re

# 输入路径（parquet 文件夹）
parquet_dir = str(PROJECT_ROOT / "shanghai_data_save" / "local_grid_time-series")

# 输出 CSV 路径
output_csv = str(PROJECT_ROOT / "shanghai_data_save" / "local_grid_time-series" / "merged_local_avg_speed.csv")

# 初始化空 DataFrame，行索引为 batch_1 到 batch_8640
index = [f"batch_{i}" for i in range(1, 8641)]

# 提取文件名中的数字部分用于排序
def extract_number(filename):
    match = re.search(r'_(\d+)\.parquet$', filename)
    return int(match.group(1)) if match else float('inf')

# 收集并排序所有 parquet 文件名
parquet_files = [
    fname for fname in os.listdir(parquet_dir)
    if fname.endswith('.parquet') and fname.startswith('honeycomb_')
]
parquet_files = sorted(parquet_files, key=extract_number)

# 用于收集所有 Series（每个网格一列）
columns = []

# 遍历排序后的文件名列表
for fname in parquet_files:
    file_path = os.path.join(parquet_dir, fname)
        
    # if fname != 'honeycomb_24102.parquet':
    #     continue
    # print(f"fname: {fname}")

    match = re.search(r'honeycomb_(\d+)\.parquet', fname)
    if not match:
        continue
    col_name = match.group(1)

    try:
        df = pd.read_parquet(file_path, columns=["batch", "local_avg_speed"])
        df = df.set_index("batch")
        df = df[~df.index.duplicated()]
        df = df.reindex(index)
        series = df["local_avg_speed"]
        series.name = col_name

        # ✅ 非空比例 < 70%，则跳过
        if series.notna().mean() < 0.7:
            continue

        columns.append(series)
        # print(f"✅ 保留并加载: {fname}")
    except Exception as e:
        print(f"❌ 加载失败 {fname}: {e}")

# 合并并输出 CSV
if columns:
    merged_df = pd.concat(columns, axis=1)
    merged_df.index.name = "batch"
    merged_df.to_csv(output_csv)
    print(f"\n✅ 已保存 CSV 文件：{output_csv}")
else:
    print("\n⚠️ 没有任何文件符合要求，未生成 CSV。")


✅ 已保存 CSV 文件：C:\DCLASS\myJupyter\paper2\shanghai_data_save\local_grid_time-series\merged_local_avg_speed.csv


In [ ]:
import pandas as pd
import numpy as np
from scipy.io import savemat

# ---------- relative CSV path ----------
file_path = (
    PROJECT_ROOT
    / "shanghai_data_save"
    / "local_grid_time-series"
    / "merged_local_avg_speed.csv"
)

df = pd.read_csv(file_path)
df

In [10]:
# 去掉 'batch' 列
data = df.drop(columns=['batch'])

# 转为 numpy 数组
data_array = data.to_numpy()  # shape: [total_batches, grid_count]

# 常量设置
num_per_day = 288
num_total = data_array.shape[0]
num_grids = data_array.shape[1]
num_days = num_total // num_per_day

In [11]:
# 只保留完整天
data_array = data_array[:num_days * num_per_day, :]

# 重新 reshape 成 [days, batches, grids]
reshaped = data_array.reshape((num_days, num_per_day, num_grids))

# 转换为目标结构：[grids, days, batches]
final_array = reshaped.transpose((2, 0, 1))  # [grid, day, batch]

In [12]:
# ---------- relative save path ----------
save_path = (
    PROJECT_ROOT
    / "shanghai_data_save"
    / "local_grid_time-series"
    / "local_speed_data.mat"
)

# 保存 .mat 文件
savemat(save_path, {'speed_data': final_array})

print(f"保存成功: {save_path}")

保存成功: C:\DCLASS\myJupyter\paper2\shanghai_data_save\local_grid_time-series\local_speed_data.mat


In [13]:
from scipy.io import loadmat
import numpy as np

# ---------- relative .mat path ----------
mat_path = (
    PROJECT_ROOT
    / "shanghai_data_save"
    / "local_grid_time-series"
    / "local_speed_data.mat"
)

data = loadmat(str(mat_path))

# 提取变量
speed_data = data['speed_data']  # shape: (grid, day, batch)

# 打印总维度
print("Shape of speed_data:", speed_data.shape)  # (格网数, 天数, 288)

# 打印第 X 个格网的所有信息（你可以设置 X）
X = 0  # 你可以改为任意索引，注意不要超过格网总数

# (X, 0): 第 X 个格网，第 0 天，288 个时间片
print(f"(X, 0): {speed_data[X, 0, :]}")

# (X, 1): 第 X 个格网，第 1 天，288 个时间片
print(f"(X, 1): {speed_data[X, 1, :]}")

# (X, 2): 第 X 个格网，第 2 天，288 个时间片
print(f"(X, 2): {speed_data[X, 2, :]}")


Shape of speed_data: (4172, 30, 288)
(X, 0): [ 4.492     6.075667 10.092     1.145     8.2245    1.217     5.04
  6.5945    9.297          nan  0.874     3.714          nan  2.432
       nan 10.258     1.923     2.488     1.465          nan       nan
       nan  1.748     1.202          nan  9.131     6.455          nan
  1.0645         nan  7.15975  14.79      8.29      3.061     9.719
  3.5175    4.506          nan 12.429          nan       nan  1.6065
       nan  8.139333  1.973          nan       nan       nan  1.475
  7.591          nan       nan       nan  3.7305         nan  0.939
       nan  4.3305         nan  6.8245         nan  1.983     6.088333
  3.239          nan  7.801          nan  7.3505    7.        2.969333
  8.613667  7.383     9.027     9.584     9.899     8.334     3.435
  6.609     6.909667  7.451     7.39975   5.998     5.773     9.630667
  8.94      4.383     3.981     6.89      2.393333 12.451    10.052
       nan  6.184     1.459          nan  8.517667      